In [1]:
%pylab inline 

import torch 
import torch.nn as nn
import torch.nn.functional as F 
import torchvision
from torchvision import transforms 
from tqdm import trange

Populating the interactive namespace from numpy and matplotlib


In [2]:
# Device Options 
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Load Data
norm_values = (0.5, 0.5, 0.5) 
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(norm_values,norm_values)
    ])
trainset = torchvision.datasets.CIFAR10(root="datasets/",download=False, train=True, transform=transform)
testset = torchvision.datasets.CIFAR10(root="datasets/", download=False, train=False, transform=transform)
trainloader = torch.utils.data.DataLoader(trainset, shuffle=True, batch_size=16)
testloader = torch.utils.data.DataLoader(testset, shuffle=False, batch_size=16)



In [10]:
# Write Training function to test on different models 
def train(model, epochs=5, adam_optim=True, lr=1e-3, mom=0, weight_decay=0):
    model.train()
    # Loss And Optimization 
    loss_function = nn.CrossEntropyLoss()
    if adam_optim:
        optim = torch.optim.Adam(model.parameters())
    else:
        optim = torch.optim.SGD(model.parameters(), lr=lr, momentum=mom, weight_decay=0)
    

    # Training Loop 
    for _ in (t := trange(epochs)):
        for x,y in trainloader:
            x,y = x.to(device), y.to(device)

            # Forward Pass 
            outputs = model(x)
            loss = loss_function(outputs, y)

            # Update Weights
            optim.zero_grad()
            loss.backward()
            optim.step()

            # Training Accuracy 
            n_correct = torch.argmax(outputs, dim=1)
            acc = (n_correct == y).float().mean() * 100
        
        t.set_description(f"acc = {acc.item():.2f}%, loss = {loss.item():.2f}")
    return model
        


      


In [4]:
# Test Model Accuracy on test set
def show_model_accuracy(model):
    model.eval()
    with torch.no_grad():
        for x,y in testloader:
            x,y = x.to(device), y.to(device)
            outputs = model(x)
            n_correct = torch.argmax(outputs, dim=1)
            acc = (n_correct == y).float().mean()*100
        print(f"Val Accuracy = {acc.item():.2f}%")
     





In [5]:
# Model Evaluation 
def eval_model(model):
    trained_model = train(model)
    acc = show_model_accuracy(trained_model)
    return acc 


In [7]:
# Model with basic convnet architecture  
class BasicNet(nn.Module):
    def __init__(self):
        super(BasicNet, self).__init__()
        self.conv1 = nn.Conv2d(3, 12, 5) # 16, 3, 28, 28 
        self.pool = nn.MaxPool2d(2,2) # 16, 12, 14, 14 
        self.conv2 = nn.Conv2d(12, 24, 5) # 16, 24, 10, 10 
        self.fc1 = nn.Linear(24*10*10, 128) 
        self.fc2 = nn.Linear(128, 64)
        self.fc3 = nn.Linear(64, 10)

    
    def forward(self, x):
        x = F.relu(self.conv1(x))
        x = self.pool(x)
        x = F.relu(self.conv2(x))
        x = x.view(-1, 24*10*10)
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = self.fc3(x)
        return x 



In [7]:
# How good is our basic convnet?
bnet = BasicNet().to(device)
eval_model(bnet)

acc = 87.50%, loss = 0.60: 100%|██████████| 5/5 [01:04<00:00, 12.90s/it]
Val Accuracy = 62.50%


In [9]:
# Resnet Cifar10 Section 4.2: https://arxiv.org/pdf/1512.03385.pdf

# Specific cfg to Cifar 10 
class Block(nn.Module):
    def __init__(self, in_size, out_size, stride=1, padding=1):
        super(Block, self).__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_size, out_size,kernel_size=3, stride=stride, padding=padding),
            nn.BatchNorm2d(out_size),
            nn.ReLU(True),

            nn.Conv2d(out_size, out_size,kernel_size=3, stride=2, padding=0),
            nn.BatchNorm2d(out_size),
            nn.ReLU(True)
        )
        self.shortcuts = IdentityPadding()
    
    def forward(self, x):
        x = self.block(x)
        x = self.shortcuts(x)
        return x 

class ResNet(nn.Module):
    def __init__(self):
        super(ResNet, self).__init__()
        # 32 X 32 
        self.init_conv = nn.Sequential(
            nn.Conv2d(3, 16, kernel_size=3, stride=1, padding=1),
            nn.BatchNorm2d(16),
            nn.ReLU(True)
        )
        
        # 8 X 8 
        self.main = nn.Sequential(
            Block(16, 16),
            Block(16, 32),
            Block(32, 64)
        )
        
        # 1 X 1 After Average Pooling 
        self.final = nn.Sequential(
            nn.AvgPool2d(8),
            nn.Linear(64, 10)
        )
    
    def forward(self, x):
        x = self.init_conv(x)
        x = self.main(x)
        x = self.final(x)
        return x 



        


# Refer: https://github.com/KellerJordan/ResNet-PyTorch-CIFAR10/blob/204803ca5be4143ee9ab4ae5e165318af45fff50/model.py#L82
class IdentityPadding(nn.Module):
    def __init__(self, num_filters, channels_in, stride):
        super(IdentityPadding, self).__init__()
        self.identity = nn.MaxPool2d(1, stride=stride)
        self.num_zeros = num_filters - channels_in
    
    def forward(self, x):
        out = F.pad(x, (0, 0, 0, 0, 0, self.num_zeros))
        out = self.identity(out)
        return out





In [ ]:
res